# Custom quadruped — Colab training

Runtime → Change runtime type → **GPU** (T4 is enough).

This trains PPO on the custom 12-DoF quadruped in MuJoCo. Physics runs on CPU (8 processes). The GPU runs PPO.

In [ ]:
import os, sys, subprocess, pathlib
from pathlib import Path

# If you uploaded the repo as a folder, point ROOT at it.
# If you cloned: git clone <your-fork> && set ROOT to that path.
ROOT = Path('/content/quad-loco')
if not ROOT.exists():
    # fallback: current directory when the notebook lives inside the repo
    here = Path.cwd()
    ROOT = here if (here / 'src' / 'quad_loco').exists() else here.parent
print('ROOT', ROOT)
assert (ROOT / 'src' / 'quad_loco').exists(), 'Upload or clone the repo so src/quad_loco is visible'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
!pip -q install mujoco gymnasium stable-baselines3 torch tensorboard onnx onnxruntime imageio imageio-ffmpeg pyyaml pytest
import torch, mujoco
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('mujoco', mujoco.__version__)

In [ ]:
from quad_loco.convert_urdf import convert
from quad_loco.paths import scene_xml
xml = convert()
print('scene', xml)
import mujoco
m = mujoco.MjModel.from_xml_path(str(xml))
print(f'nq={m.nq} nu={m.nu}')

In [ ]:
from quad_loco.env import QuadrupedVelocityEnv
env = QuadrupedVelocityEnv(easy=True, command=(0.4, 0.0, 0.0))
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'cmd', info['command'])
for _ in range(20):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
print('height', info['base_height'], 'reward', r)
env.close()

In [ ]:
# 1.5M steps, ~2-4 hours on Colab T4. Drop --timesteps for a 50k smoke run.
!python scripts/train.py --config configs/ppo_colab.yaml --run-name colab_easy

In [ ]:
!python scripts/eval.py --model logs/colab_easy/final_model.zip --easy --command 0.5 0 0 --video videos/walk.mp4 --episodes 1
!python scripts/export_onnx.py --model logs/colab_easy/final_model.zip --out logs/colab_easy/policy.onnx
from google.colab import files
for p in ['logs/colab_easy/final_model.zip', 'logs/colab_easy/vecnormalize.pkl', 'logs/colab_easy/policy.onnx', 'videos/walk.mp4']:
    if Path(p).exists():
        files.download(p)